In [9]:
import sys
import os

sys.path.append(os.path.abspath(".."))

from config.spark import spark


In [10]:
from pyspark.sql.types import StructType,StructField,StringType,IntegerType,TimestampType,DoubleType

order_items_schema = StructType([
    StructField("order_id",StringType()),
    StructField("order_item_id",IntegerType()),
    StructField("product_id",StringType()),
    StructField("seller_id",StringType()),
    StructField("shipping_limit_date",TimestampType()),
    StructField("price",DoubleType()),
    StructField("freight_value",DoubleType()),
])


In [11]:
order_items_df = (
    spark.read.format('csv')
    .option('header','True')
   # .option('inferschema','True')
   .schema(order_items_schema)
    .load('../landing/files/olist_order_items_dataset.csv')
    
)

In [12]:
from pyspark.sql.functions import current_timestamp

order_items_df = order_items_df.withColumn("ingestion_time",current_timestamp())

In [13]:
order_items_df.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- order_item_id: integer (nullable = true)
 |-- product_id: string (nullable = true)
 |-- seller_id: string (nullable = true)
 |-- shipping_limit_date: timestamp (nullable = true)
 |-- price: double (nullable = true)
 |-- freight_value: double (nullable = true)
 |-- ingestion_time: timestamp (nullable = false)



In [14]:
order_items_df.write.mode("overwrite").saveAsTable("bronze.order_items")

In [15]:
spark.sql(
    """
    select * from bronze.order_items
    """
).show(100)

+--------------------+-------------+--------------------+--------------------+-------------------+------+-------------+--------------------+
|            order_id|order_item_id|          product_id|           seller_id|shipping_limit_date| price|freight_value|      ingestion_time|
+--------------------+-------------+--------------------+--------------------+-------------------+------+-------------+--------------------+
|8ac26cb701a7887cc...|            1|4ebb87ba41ca44632...|7a67c85e85bb2ce85...|2017-05-22 16:05:14|109.99|        18.02|2026-10-17 14:42:...|
|8ac2728285fd4228f...|            1|8b90be4893a4277a9...|004c9cd9d87a3c30c...|2017-03-15 14:09:17|109.99|         8.27|2026-10-17 14:42:...|
|8ac2728285fd4228f...|            2|fa94f25a73969e3a2...|004c9cd9d87a3c30c...|2017-03-15 14:09:17|109.99|        16.55|2026-10-17 14:42:...|
|8ac2728285fd4228f...|            3|b01cedfa96d891427...|004c9cd9d87a3c30c...|2017-03-15 14:09:17|259.99|        21.01|2026-10-17 14:42:...|
|8ac2728285fd